# REDACT — Constitution Generation

**Stage 1 of 3.** Generates a structured category hierarchy for constitutional
classifier training, then expands each constitution entry into full realistic input
prompts.

For each taxonomy category, the constitution spans 4 severity levels:
- **Harmful** — absolutely harmful, always flag
- **Dual-use harmful** — borderline harmful framing, debatable
- **Dual-use benign** — borderline benign framing, could look harmful
- **Benign** — absolutely benign, never flag (hard negatives)

**Handoff artifact:** `Datasets/constitution_inputs_merged.csv` — a single clean CSV of
expanded prompts that the jailbreak notebook (`jailbreak_augmentation.ipynb`) can load
whole.

Requires `ANTHROPIC_API_KEY` (constitution generation, Claude Opus) and `VENICE_API_KEY`
(input expansion). Set them in the environment or a `.env` file.

In [ ]:
from redact import set_seed
set_seed(42)

## Configuration

Adjust these settings before running. Small values are set for demo purposes.

In [ ]:
# --- Models ---
MODEL = "venice-uncensored-vllm"        # Input-expansion generation model
CONSTITUTION_MODEL = "claude-opus-4-6"  # Constitution generation model (Claude Opus)
TAXONOMY = "content_moderation_categories"

# --- Constitution ---
CONSTITUTION_CATEGORIES = 5            # Constitution categories per type per taxonomy category
CONSTITUTION_ENTRY_TYPES = None       # None = all 4 (harmful, benign, dual_use_benign, dual_use_harmful)
CONSTITUTION_TAXONOMY_CATEGORIES = None  # None = all taxonomy categories
CONSTITUTION_STANDALONE_BENIGN = True    # Generate category-free benign entries (single LLM call)
CONSTITUTION_STANDALONE_BENIGN_CATEGORIES = 10

# --- Constitution-to-input ---
CONSTITUTION_INPUT_STYLE = ["long", "short"]  # "long" (2-5 sentences) and/or "short" (5-20 words)
CONSTITUTION_INPUT_SAMPLES_PER_ENTRY = 2      # Prompts to generate per constitution entry
CONSTITUTION_INPUT_ENTRY_TYPES = None         # Which entry types to expand (None = all)
CONSTITUTION_INPUT_BATCH_SIZE = 32            # Entries per LLM engine pass

## Step 1: Generate Constitution

For each taxonomy category, generates entries across the 4 severity levels using
Claude Opus. Optionally generates **standalone benign** entries in a single
category-free LLM call (no taxonomy influence), saved separately to `general_benign.csv`.

Entries are saved to `Data_cache/constitution/`. Each entry later seeds N input prompts.

In [ ]:
from redact import generate_constitution

constitution = generate_constitution(
    taxonomy=TAXONOMY,
    entry_types=CONSTITUTION_ENTRY_TYPES,
    num_categories=CONSTITUTION_CATEGORIES,
    model=CONSTITUTION_MODEL,
    num_taxonomy_categories=CONSTITUTION_TAXONOMY_CATEGORIES,
    include_standalone_benign=CONSTITUTION_STANDALONE_BENIGN,
    standalone_benign_categories=CONSTITUTION_STANDALONE_BENIGN_CATEGORIES,
)

print(f"\nGenerated {len(constitution)} constitution entries")
if not constitution.empty:
    print("\n=== By Entry Type ===")
    print(constitution["entry_type"].value_counts().to_string())
    print("\n=== By Source Category ===")
    print(constitution["source_category"].value_counts().to_string())
constitution.head(10)

## Step 2: Constitution-to-Input Generation

Expands each constitution entry's short `sample_description` into
`CONSTITUTION_INPUT_SAMPLES_PER_ENTRY` full realistic prompts, using the composable
`generate_inputs(constitution_df=...)` API (constitution-seeded mode).

Two template styles are available:
- **long** — detailed, multi-sentence prompts (2-5 sentences)
- **short** — concise, direct prompts (5-20 words)

Generation and checking are both batched per `batch_size`. We pass an explicit
`dataset_dir` so constitution inputs land under `Datasets/constitution_inputs/{category}/`,
isolated from standalone content-moderation inputs.

> Note: `generate_inputs_from_constitution()` is deprecated — prefer
> `generate_inputs(constitution_df=...)` as below.

In [ ]:
import pandas as pd
from redact import generate_inputs, get_output_dir

CONSTITUTION_INPUTS_DIR = get_output_dir() / "Datasets" / "constitution_inputs"

styles = (
    CONSTITUTION_INPUT_STYLE
    if isinstance(CONSTITUTION_INPUT_STYLE, list)
    else [CONSTITUTION_INPUT_STYLE]
)
all_constitution_inputs = []

for style in styles:
    result = generate_inputs(
        constitution_df=constitution,        # triggers constitution-seeded mode
        style=style,
        samples_per_entry=CONSTITUTION_INPUT_SAMPLES_PER_ENTRY,
        entry_types=CONSTITUTION_INPUT_ENTRY_TYPES,
        model=MODEL,
        dataset_dir=CONSTITUTION_INPUTS_DIR,
        batch_size=CONSTITUTION_INPUT_BATCH_SIZE,
    )
    print(f"\n[{style}] Generated {len(result)} accepted prompts")
    all_constitution_inputs.append(result)

constitution_inputs = (
    pd.concat(all_constitution_inputs, ignore_index=True)
    if all_constitution_inputs
    else pd.DataFrame()
)
print(f"\nTotal: {len(constitution_inputs)} accepted prompts from constitution entries")
constitution_inputs.head(10)

## Step 3: Merge → Handoff Dataset

Merges all per-category CSVs from `Datasets/constitution_inputs/` into a single clean CSV.
Filters out rejected samples, drops generation-only columns, and renames `sample` →
`prompt`. Saves to `Datasets/constitution_inputs_merged.csv` — the artifact the jailbreak
notebook consumes.

In [ ]:
from redact.dataset import merge_constitution_input_csvs

merged_path = get_output_dir() / "Datasets" / "constitution_inputs_merged.csv"

constitution_inputs_merged = merge_constitution_input_csvs(
    base_dir=CONSTITUTION_INPUTS_DIR,
    output_path=merged_path,
)

print(f"Merged {len(constitution_inputs_merged)} accepted samples -> {merged_path}")
print(f"Columns: {list(constitution_inputs_merged.columns)}")
if not constitution_inputs_merged.empty:
    print("\n=== By Category ===")
    print(constitution_inputs_merged["category"].value_counts().to_string())
    print("\n=== By Entry Type ===")
    print(constitution_inputs_merged["entry_type"].value_counts().to_string())
constitution_inputs_merged.head(10)